# 💠 01-数据结构与算法 · 主题 12：位运算

> 位运算题量不大但**一题一思路**（异或去重、清除最低位、集合枚举），会了就是送分题。
> 本 notebook 目标：**背熟 6 个 trick → 吃透 8 道高频题 → 通过 25 道自测**。

**前置**：无强依赖。**运行环境**：Python 3.8+。

## §1 位运算基础（背熟）

| 运算符 | 含义 | 例子 |
|--------|------|------|
| `&` | 按位与 | `5 & 3 = 1`（101 & 011） |
| `\|` | 按位或 | `5 \| 3 = 7` |
| `^` | 按位异或 | `5 ^ 3 = 6` |
| `~` | 取反 | `~5 = -6`（补码） |
| `<<` | 左移（×2） | `1 << 3 = 8` |
| `>>` | 右移（÷2） | `8 >> 2 = 2` |

### 六大 trick（面试核心）

1. **异或自反**：$a \oplus a = 0$，$a \oplus 0 = a$ → 成对出现可抵消
2. **清除最低位 1**：`x & (x - 1)` → 把最右边的 1 变成 0（数 1 个数、判 2 的幂）
3. **取最低位 1**：`x & (-x)` → 得到只含最低位 1 的数（树状数组、lowbit）
4. **判第 k 位**：`(x >> k) & 1`
5. **置位/清位**：`x | (1 << k)`（置 1）；`x & ~(1 << k)`（清 0）
6. **交换不用临时变量**：`a ^= b; b ^= a; a ^= b`（面试题梗，实际工程不用）

### Python 注意

- 整数**无限位宽**，负数用**补码**（`-x = ~x + 1`）
- `x & (x-1)` 对负数同样适用（补码下有意义）
- Python 有内置 `bin(x).count('1')`，但面试要求手写

## §2 只出现一次的数字（LeetCode 136）：异或

**题目**：非空数组，只有一个元素出现 1 次，其余恰好 2 次，找出它。

**公式**：全体异或：

$$(a_1 \oplus a_2 \oplus \cdots \oplus a_n) = \text{只出现一次的数}$$

因为每对相同数异或为 0，0 ⊕ x = x。

**复杂度**：$O(n)$ 时间，$O(1)$ 空间（对比哈希 $O(n)$ 空间）。

> 💡 变体一：137 每个数出现 3 次——用位计数模 3（`(sum(第k位1的个数) % 3) << k` 组装）。
> 💡 变体二：260 有两个唯一数——先全体异或得到 `a^b`，取最低位 1 分组再分别异或。

In [ ]:
def single_number(nums):
    res = 0
    for x in nums:
        res ^= x
    return res

def single_numbers_260(nums):
    # 两个唯一数 a, b
    x = 0
    for v in nums:
        x ^= v                  # x = a ^ b
    mask = x & (-x)             # 最低位 1：a、b 在这一位不同
    a = b = 0
    for v in nums:
        if v & mask:            # 按该位分组
            a ^= v
        else:
            b ^= v
    return [a, b]

print('136:', single_number([4, 1, 2, 1, 2]))        # 期望 4
print('136:', single_number([2, 2, 1]))             # 期望 1
print('260:', sorted(single_numbers_260([1, 2, 1, 3, 2, 5])))  # 期望 [3, 5]

## §3 位 1 的个数（191）/ 2 的幂（231）/ 缺失数字（268）

### 191 汉明重量（数 1 的个数）

**trick**：`n = n & (n - 1)` 每次消掉一个 1：

$$\text{popcount}(n) = \#\{ \text{循环} n \& (n-1) \text{直到} n = 0 \}$$

**复杂度**：$O(\text{位数中 1 的个数})$ ≤ $O(32/64)$。

### 231 2 的幂

$$n > 0 \ \land \ (n \& (n - 1)) = 0$$

因为 2 的幂二进制只有一个 1（`1, 10, 100, 1000...`）。

### 268 缺失数字

**思路**：`[0..n]` 全部异或上数组所有数，成对的抵消，剩下的就是缺失值：

$$(0 \oplus 1 \oplus \cdots \oplus n) \oplus (\text{nums 全体}) = \text{缺失值}$$

In [ ]:
def hamming_weight(n):
    cnt = 0
    while n:
        n &= n - 1              # 清除最低位 1
        cnt += 1
    return cnt

def is_power_of_two(n):
    return n > 0 and (n & (n - 1)) == 0

def missing_number(nums):
    res = 0
    n = len(nums)
    for i in range(n + 1):
        res ^= i                # 0..n 全异或
    for x in nums:
        res ^= x                # 再异或数组
    return res

print('191:', hamming_weight(11))         # 期望 3（1011）
print('191:', hamming_weight(0))          # 期望 0
print('231:', is_power_of_two(16), is_power_of_two(18), is_power_of_two(1), is_power_of_two(0))
print('     期望 True False True False')
print('268:', missing_number([3, 0, 1]))               # 期望 2
print('268:', missing_number([0, 1]))                  # 期望 2
print('268:', missing_number([9, 6, 4, 2, 3, 5, 7, 0, 1]))  # 期望 8

## §4 比特位计数（338）：位运算 + DP 组合

**题目**：返回 `0..n` 每个数的二进制 1 个数。

**DP 转移（核心）**：右移一位 + 最低位信息：

$$\text{popcount}(i) = \text{popcount}(i >> 1) + (i \& 1)$$

因为右移把最低位丢掉，`i & 1` 补回它。

**复杂度**：$O(n)$ 时间（比逐个 popcount $O(n \log n)$ 优），$O(1)$ 额外空间。

> 💡 另一个等价转移：`popcount(i) = popcount(i & (i-1)) + 1`。

In [ ]:
def count_bits(n):
    dp = [0] * (n + 1)
    for i in range(1, n + 1):
        dp[i] = dp[i >> 1] + (i & 1)     # 去掉最低位 + 补回最低位
    return dp

def count_bits_v2(n):
    dp = [0] * (n + 1)
    for i in range(1, n + 1):
        dp[i] = dp[i & (i - 1)] + 1      # 清除一个 1 再补 1
    return dp

print('338:', count_bits(5))       # 期望 [0, 1, 1, 2, 1, 2]
print('338:', count_bits(0))       # 期望 [0]
print('v2 一致:', count_bits_v2(8) == count_bits(8))

## §5 汉明距离（461）/ 颠倒二进制位（190）

### 461 汉明距离（两数二进制不同的位数）

**公式**：

$$\text{dist}(x, y) = \text{popcount}(x \oplus y)$$

异或后不同的位都为 1，数 1 即可。

### 190 颠倒二进制位（反转 32 位）

**思路**：结果左移腾位 + 取原数最低位：

```python
res = 0
for _ in range(32):
    res = (res << 1) | (n & 1)   # 原最低位移入结果
    n >>= 1
return res
```

In [ ]:
def hamming_distance(x, y):
    return hamming_weight(x ^ y)

def reverse_bits(n):
    res = 0
    for _ in range(32):
        res = (res << 1) | (n & 1)
        n >>= 1
    return res

print('461:', hamming_distance(1, 4))      # 期望 2（001 vs 100）
print('461:', hamming_distance(3, 1))      # 期望 1
print('190:', reverse_bits(43261596))      # 期望 964176192
print('190(0):', reverse_bits(0))          # 期望 0
print('190(全1):', reverse_bits(0xFFFFFFFF))  # 期望 0xFFFFFFFF

## §6 集合枚举与状态压缩（位运算扩展）

### 子集枚举

n 个元素的任意子集 ↔ 一个 n 位二进制数：第 i 位 1 = 选第 i 个元素：

$$\text{子集个数} = 2^n \ (\text{mask} = 0 \dots 2^n - 1)$$

```python
for mask in range(1 << n):
    subset = [nums[i] for i in range(n) if (mask >> i) & 1]
```

### 状态压缩 DP 入门

状态 = visited 集合（`mask & (1 << i)` 判是否访问过 i），经典：旅行商、划分为 k 个相等子集（698）。

**复杂度提示**：$2^n$ 个状态 × 每状态 $O(n)$ 转移 → $O(n \cdot 2^n)$，n ≤ 20 时可接受。

In [ ]:
def subsets_by_bit(nums):
    # 位枚举版子集（对照主题 7 的回溯版）
    n = len(nums)
    res = []
    for mask in range(1 << n):
        sub = []
        for i in range(n):
            if (mask >> i) & 1:
                sub.append(nums[i])
        res.append(sub)
    return res

s = subsets_by_bit([1, 2, 3])
print('位枚举子集个数:', len(s))     # 期望 8
print('按 mask 序:', s)
expected = [[], [1], [2], [1, 2], [3], [1, 3], [2, 3], [1, 2, 3]]
print('与回溯版结果集一致:', sorted(map(tuple, s)) == sorted(map(tuple, expected)))  # 期望 True


## §7 常见 bug 与边界（面试踩坑区）

- **异或结果混淆**：`a ^ b` 是「不同位为 1」，不是「相加/相减」；计数要用 popcount
- **负数右移**：Python `n >>= 1` 对负数做**算术右移**（补 1），数位个数的题先把负数转正或改用 `n & (n-1)` 法
- **`x & -x` 在 Python**：无限位宽下 `-x = ~x + 1`，`x & (-x)` 仍正确取最低位 1（正数场景）
- **位宽假设**：`1 << 31` 在 Python 无溢出（任意大），但 LeetCode 190 明确 32 位——循环固定 32 次
- **运算符优先级**：`(mask >> i) & 1` 必须加括号，`&` 优先级低于 `==`
- **`~x` 陷阱**：取反结果带符号（`~5 = -6`），清位要写成 `x & ~(1 << k)` 而不是 `x & !(...)`

## §8 面试追问与扩展（加分项）

### 8.1 为什么负数用补码

补码使加减法与位运算统一（`x + (-x) = 0` 自动成立），硬件只需加法器；`-x = ~x + 1`。

### 8.2 交换论证：`a ^= b; b ^= a; a ^= b`

三步分别得到 `a'=a^b`、`b'=a`、`a''=b`——利用异或自反。

### 8.3 位运算在工程中的场景

- 权限位掩码（read/write/execute）
- 哈希散列（`hash % capacity` 优化为 `& (cap-1)`）
- 图像 RGBA 通道打包/解包
- 快速乘以 2 的幂（`<<`）——注意可读性优先，工程上不滥用

## §9 自测清单（L1 必会 / L2 进阶 / L3 挑战）

### L1 必会（10 题）

- [ ] 1. 只出现一次的数字（LeetCode 136）
- [ ] 2. 位 1 的个数（191）
- [ ] 3. 2 的幂（231）
- [ ] 4. 缺失数字（268）
- [ ] 5. 汉明距离（461）
- [ ] 6. 四的幂（342）
- [ ] 7. 交替位二进制数（693）
- [ ] 8. 数字的补数（476）
- [ ] 9. 二进制求和（67）
- [ ] 10. 颠倒二进制位（190）

### L2 进阶（10 题）

- [ ] 11. 只出现一次的数字 II（137）
- [ ] 12. 只出现一次的数字 III（260）
- [ ] 13. 比特位计数（338）
- [ ] 14. 两整数之和（371，位运算加法）
- [ ] 15. 子集（78，位枚举法）
- [ ] 16. 二进制中 1 的个数（剑指 Offer 15，面试常问）
- [ ] 17. UTF-8 编码验证（393）
- [ ] 18. 数字范围按位与（201）
- [ ] 19. 最大单词长度乘积（318，位掩码）
- [ ] 20. 转换为 16 进制数（405）

### L3 挑战（5 题）

- [ ] 21. 位运算实现加减乘除（无乘除号做除法 29）
- [ ] 22. 划分为 k 个相等的子集（698，状压）
- [ ] 23. 最大的异或（421，字典树+位）
- [ ] 24. 戳气球（312 的状压视角）
- [ ] 25. 最短超级串（943，TSP 状压）

### 自测要点

- 3 分钟内盲写：136 + 191 + 231
- 讲清 `x & (x-1)` 与 `x & -x` 的区别与用途
- 能手推 338 的 DP 转移公式

## 附录：参考与下节预告

- 《算法竞赛入门经典》位运算章节
- Python 官方整数文档（任意精度、位运算语义）

### 下节预告：主题 13 — 字符串（收尾）

- 反转/匹配/回文/计数四大类 + KMP 与滚动哈希概念

> 💡 本主题验收：`高频题单.md`「位运算」一节全部打勾。

## §10 高频补充：只出现一次的数字 II / 两整数之和 / 数字范围按位与 / 二进制求和

### 137 只出现一次的数字 II（其余元素各出现 3 次）
- 异或只能解决"出现 2 次"（奇偶抵消）；出现 3 次时需**逐位统计**：
  对所有数字的某一位统计 1 的个数，`cnt % 3` 即为唯一数在该位的值。
- 实现：循环 32 位，`cnt = sum((x >> i) & 1 for x in nums)`，`cnt % 3 == 1` 则置位。
- Python 注意：整数任意精度、负数补码视为无限长，用 `mask = (1 << 32) - 1` 截断到 32 位，
  最后若符号位（第 31 位）为 1 则减 `2**32` 还原成负数。


In [ ]:
def single_number_ii(nums):
    # 位计数模 3：对每一位统计 1 的出现次数 % 3，组装出唯一数
    res = 0
    for i in range(32):
        cnt = sum((x >> i) & 1 for x in nums)   # 第 i 位上 1 的个数
        if cnt % 3:                              # 出现 3 次的元素在此位会被 3 整除
            res |= 1 << i
    if res >= (1 << 31):                         # 符号位为 1 → 负数还原
        res -= 1 << 32
    return res

print('137:', single_number_ii([2, 2, 3, 2]))                 # 期望 3
print('137:', single_number_ii([0, 1, 0, 1, 0, 1, 99]))       # 期望 99
print('137 负数:', single_number_ii([-2, -2, 1, -2]))         # 期望 1
print('137 负数:', single_number_ii([1, 1, 1, -3]))           # 期望 -3

In [ ]:
def get_sum(a, b):
    # 位运算加法：a ^ b = 无进位和，(a & b) << 1 = 进位；迭代到进位为 0
    MASK = 0xFFFFFFFF                    # 32 位掩码，模拟补码截断
    a &= MASK
    b &= MASK
    while b:
        carry = (a & b) << 1             # 进位
        a = (a ^ b) & MASK               # 无进位和
        b = carry & MASK
    return a if a <= 0x7FFFFFFF else a - (1 << 32)   # 符号还原

print('371:', get_sum(1, 2))                     # 期望 3
print('371:', get_sum(-1, 2))                    # 期望 1
print('371 溢出:', get_sum(2147483647, 1))       # 期望 -2147483648（32 位溢出回绕）
print('371 溢出:', get_sum(-2147483648, -1))     # 期望 2147483647

In [ ]:
def range_bitwise_and(m, n):
    # 公共前缀法：[m, n] 区间按位与 = m、n 的公共二进制前缀补 0
    # m、n 同时右移直到相等，记录移位次数，再把公共前缀左移回来
    shift = 0
    while m < n:
        m >>= 1
        n >>= 1
        shift += 1
    return m << shift

print('201:', range_bitwise_and(5, 7))             # 期望 4（101 & 110 & 111）
print('201:', range_bitwise_and(0, 0))             # 期望 0
print('201 大数:', range_bitwise_and(1, 2147483647))  # 期望 0
print('201:', range_bitwise_and(12, 15))           # 期望 12

In [ ]:
def add_binary(a, b):
    # 手写二进制字符串相加：从低位到高位模拟进位，最后反转结果
    i, j = len(a) - 1, len(b) - 1
    carry = 0
    res = []
    while i >= 0 or j >= 0 or carry:
        s = carry
        if i >= 0:
            s += int(a[i]); i -= 1
        if j >= 0:
            s += int(b[j]); j -= 1
        res.append(str(s & 1))       # 当前位
        carry = s >> 1               # 进位
    return ''.join(reversed(res))

print('67:', add_binary('11', '1'))           # 期望 100
print('67:', add_binary('1010', '1011'))      # 期望 10101
print('67:', add_binary('111', '111'))        # 期望 1110
print('67 对照:', add_binary('1010', '1011') == bin(10 + 11)[2:])   # 期望 True